<a href="https://colab.research.google.com/github/sl007ha/qqq-risk-monitor/blob/phase2-v32-clean/notebooks/20_phase2_clean_build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy scikit-learn matplotlib requests fredapi openpyxl xlrd yfinance --quiet

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import yfinance as yf

from fredapi import Fred
from google.colab import userdata

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    average_precision_score
)

FRED_API_KEY = userdata.get("FRED_API_KEY")
fred = Fred(api_key=FRED_API_KEY)

In [3]:
ndx = yf.download("^NDX", start="1985-01-01", auto_adjust=False, progress=False)

if isinstance(ndx.columns, pd.MultiIndex):
    ndx.columns = ndx.columns.get_level_values(0)

ndx_d = ndx[["Close"]].rename(columns={"Close": "ndx_close"}).dropna()

print(ndx_d.head())
print(ndx_d.tail())

Price        ndx_close
Date                  
1985-10-01  112.139999
1985-10-02  110.824997
1985-10-03  110.870003
1985-10-04  110.074997
1985-10-07  108.199997
Price          ndx_close
Date                    
2026-05-04  27651.820312
2026-05-05  28015.060547
2026-05-06  28599.169922
2026-05-07  28563.949219
2026-05-08  29234.990234


In [4]:
print("NDX daily range:", ndx_d.index.min(), "to", ndx_d.index.max())

NDX daily range: 1985-10-01 00:00:00 to 2026-05-08 00:00:00


In [5]:
px = ndx_d.copy()
px["ret_d"] = px["ndx_close"].pct_change()

px_m = px[["ndx_close"]].resample("ME").last()

px_m["ret_1m"] = px_m["ndx_close"].pct_change(1)
px_m["ret_3m"] = px_m["ndx_close"].pct_change(3)
px_m["ret_6m"] = px_m["ndx_close"].pct_change(6)
px_m["tsmom_12m"] = px_m["ndx_close"].pct_change(12)

px_m["ma_200d"] = px["ndx_close"].rolling(200).mean().resample("ME").last()
px_m["dist_200dma"] = px_m["ndx_close"] / px_m["ma_200d"] - 1

px_m["rolling_6m_high"] = px_m["ndx_close"].rolling(6, min_periods=3).max()
px_m["rolling_12m_high"] = px_m["ndx_close"].rolling(12, min_periods=6).max()

px_m["dd_from_6m_high"] = px_m["ndx_close"] / px_m["rolling_6m_high"] - 1
px_m["dd_from_12m_high"] = px_m["ndx_close"] / px_m["rolling_12m_high"] - 1

px["vol_3m_daily"] = px["ret_d"].rolling(63).std() * np.sqrt(252)
px["vol_12m_daily"] = px["ret_d"].rolling(252).std() * np.sqrt(252)

vol_m = px[["vol_3m_daily", "vol_12m_daily"]].resample("ME").last()
vol_m.columns = ["vol_3m", "vol_12m"]
vol_m["vol_ratio_3m_12m"] = vol_m["vol_3m"] / vol_m["vol_12m"]

price_features = px_m[
    [
        "ndx_close",
        "dist_200dma",
        "tsmom_12m",
        "ret_1m",
        "ret_3m",
        "ret_6m",
        "dd_from_6m_high",
        "dd_from_12m_high",
    ]
].join(
    vol_m[["vol_3m", "vol_ratio_3m_12m"]],
    how="left"
)

print(price_features.tail())

               ndx_close  dist_200dma  tsmom_12m    ret_1m    ret_3m  \
Date                                                                   
2026-01-31  25552.390625     0.084099   0.189698  0.011982 -0.011824   
2026-02-28  24960.039062     0.035420   0.195152 -0.023182 -0.018669   
2026-03-31  23740.189453    -0.028048   0.231437 -0.048872 -0.059789   
2026-04-30  27452.119141     0.106952   0.402692  0.156356  0.074346   
2026-05-31  29234.990234     0.171446   0.369898  0.064945  0.171272   

              ret_6m  dd_from_6m_high  dd_from_12m_high    vol_3m  \
Date                                                                
2026-01-31  0.100537        -0.011824         -0.011824  0.164192   
2026-02-28  0.065966        -0.034732         -0.034732  0.150317   
2026-03-31 -0.038079        -0.081906         -0.081906  0.182558   
2026-04-30  0.061644         0.000000          0.000000  0.199013   
2026-05-31  0.149405         0.000000          0.000000  0.193833   

           

In [7]:
yc_10y3m = fred.get_series("T10Y3M", observation_start="1985-01-01")
yc_10y3m = yc_10y3m.resample("ME").last().to_frame("yc_10y3m")

cfnai_ma3 = fred.get_series("CFNAIMA3", observation_start="1985-01-01")
cfnai_ma3 = cfnai_ma3.resample("ME").last().to_frame("cfnai_ma3")

macro_features = yc_10y3m.join(cfnai_ma3, how="outer")

print(macro_features.tail())

            yc_10y3m  cfnai_ma3
2026-01-31      0.59      -0.07
2026-02-28      0.30       0.03
2026-03-31      0.60      -0.03
2026-04-30      0.72        NaN
2026-05-31      0.69        NaN


In [14]:
!pip install xlrd openpyxl requests --quiet

import pandas as pd
import numpy as np
import requests

In [16]:
# ============================================================
# Step H: Download Shiller CAPE and generate cape_z
# Stable version: Shiller Data sheet header is row 7
# ============================================================

!pip install xlrd openpyxl requests --quiet

import pandas as pd
import numpy as np
import requests

SHILLER_URL = "https://img1.wsimg.com/blobby/go/e5e77e0b-59d1-44d9-ab25-4763ac982e53/downloads/441f0d2c-37e4-4803-b4e2-8fe10407fbf6/ie_data.xls?ver=1778098504874"
SHILLER_LOCAL_PATH = "/content/ie_data.xls"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/vnd.ms-excel,application/octet-stream,*/*",
}

r = requests.get(SHILLER_URL, headers=headers, timeout=30)

print("HTTP status:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("File size KB:", round(len(r.content) / 1024, 1))

if r.status_code != 200:
    raise RuntimeError(f"Shiller download failed. HTTP status = {r.status_code}")

head_text = r.content[:500].decode(errors="ignore").lower()
if "<html" in head_text and "excel" not in head_text:
    raise RuntimeError("Downloaded content looks like HTML, not Excel. Please check Shiller URL.")

with open(SHILLER_LOCAL_PATH, "wb") as f:
    f.write(r.content)

print(f"Saved to {SHILLER_LOCAL_PATH}")


# ===== Read Shiller data =====
# 关键修复：
# Shiller 的 Data sheet 前 7 行是说明/多层表头，第 8 行才是真正字段名
# 所以这里直接 skiprows=7
shiller = pd.read_excel(
    SHILLER_LOCAL_PATH,
    sheet_name="Data",
    skiprows=7
)

# 清理列名
shiller.columns = [str(c).strip() for c in shiller.columns]

print("Columns:")
print(shiller.columns.tolist())

print("\nPreview:")
print(shiller.head())


# ===== Basic validation =====
if "Date" not in shiller.columns:
    raise ValueError(f"'Date' column not found. Columns = {shiller.columns.tolist()}")

# 优先使用精确的 CAPE 列，而不是 TR CAPE / Excess CAPE Yield
if "CAPE" in shiller.columns:
    cape_col = "CAPE"
else:
    # fallback：按位置取第 13 列，Shiller 表里 CAPE 通常是第 12 index / 第13列
    # 仅当列名识别失败时使用
    print("Exact 'CAPE' column not found. Falling back to positional column index 12.")
    cape_col = shiller.columns[12]

print("Using CAPE column:", cape_col)


# ===== Drop invalid rows =====
shiller = shiller.dropna(subset=["Date"])
shiller = shiller[shiller["Date"].apply(lambda x: isinstance(x, (int, float, np.integer, np.floating)))]


# ===== Convert Shiller date format: 1871.01 -> 1871-01 month end =====
def shiller_date_to_timestamp(d):
    year = int(d)
    month = int(round((float(d) - year) * 100))

    if month < 1:
        month = 1
    if month > 12:
        month = 12

    return pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)


shiller["date"] = shiller["Date"].apply(shiller_date_to_timestamp)
shiller = shiller.set_index("date").sort_index()


# ===== Extract CAPE =====
cape_features = shiller[[cape_col]].copy()
cape_features.columns = ["cape"]
cape_features["cape"] = pd.to_numeric(cape_features["cape"], errors="coerce")
cape_features = cape_features.dropna()

# 去掉重复 index，如果有
cape_features = cape_features[~cape_features.index.duplicated(keep="last")]


# ===== Calculate 20-year rolling z-score =====
# 这里用 240 个月滚动窗口，比 expanding 更符合你原 Phase 2 spec
window = 240
min_periods = 120

cape_features["cape_mean_20y"] = cape_features["cape"].rolling(
    window,
    min_periods=min_periods
).mean()

cape_features["cape_std_20y"] = cape_features["cape"].rolling(
    window,
    min_periods=min_periods
).std()

cape_features["cape_z"] = (
    (cape_features["cape"] - cape_features["cape_mean_20y"])
    / cape_features["cape_std_20y"]
)

cape_features = cape_features[["cape", "cape_z"]].dropna()


# ===== Validation checks =====
print("\n=== CAPE features tail ===")
print(cape_features.tail(12))

print("\n=== CAPE feature summary ===")
print(cape_features.describe().round(4))

print("\n=== Key periods check ===")
print("Dot-com:")
print(cape_features.loc["1999-06":"2000-06"].tail(12))

print("\nGFC pre-crisis:")
print(cape_features.loc["2007-01":"2007-12"])

print("\nRecent:")
print(cape_features.tail(6))

cape_features.to_csv("/content/cape_features.csv")
print("\nSaved to /content/cape_features.csv")

HTTP status: 200
Content-Type: application/vnd.ms-excel
File size KB: 1621.0
Saved to /content/ie_data.xls
Columns:
['Date', 'P', 'D', 'E', 'CPI', 'Fraction', 'Rate GS10', 'Price', 'Dividend', 'Price.1', 'Earnings', 'Earnings.1', 'CAPE', 'Unnamed: 13', 'TR CAPE', 'Unnamed: 15', 'Yield', 'Returns', 'Returns.1', 'Real Return', 'Real Return.1', 'Returns.2']

Preview:
      Date     P     D    E        CPI     Fraction Rate GS10       Price  \
0  1871.01  4.44  0.26  0.4  12.464061  1871.041667      5.32  118.545708   
1  1871.02   4.5  0.26  0.4  12.844641  1871.125000  5.323333  116.587763   
2  1871.03  4.61  0.26  0.4  13.034972  1871.208333  5.326667  117.693713   
3  1871.04  4.74  0.26  0.4  12.559226  1871.291667      5.33  125.596602   
4  1871.05  4.86  0.26  0.4  12.273812  1871.375000  5.333333  131.770822   

   Dividend     Price.1  ...  CAPE  Unnamed: 13  TR CAPE  Unnamed: 15  Yield  \
0  6.941866  118.545708  ...   NaN          NaN      NaN          NaN    NaN   
1  6.73618

In [20]:
# ============================================================
# Step I: Load and clean AAII sentiment from GitHub cache
# ============================================================

!pip install requests xlrd openpyxl --quiet

import os
import pandas as pd
import requests

AAII_CACHE_URL = "https://raw.githubusercontent.com/sl007ha/qqq-risk-monitor/phase2-v32-clean/data/raw/sentiment.xls"
LOCAL_AAII = "/content/sentiment.xls"


def looks_like_excel(content: bytes) -> bool:
    if content is None or len(content) < 1000:
        return False

    head_text = content[:1000].decode(errors="ignore").lower()

    if "<html" in head_text:
        return False
    if "access denied" in head_text:
        return False
    if "forbidden" in head_text:
        return False
    if "github.com/login" in head_text:
        return False

    return True


headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/vnd.ms-excel,application/octet-stream,*/*",
}

r = requests.get(AAII_CACHE_URL, headers=headers, timeout=30)

print("HTTP status:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("File size KB:", round(len(r.content) / 1024, 1))

if r.status_code != 200:
    raise RuntimeError(f"GitHub AAII cache download failed: HTTP {r.status_code}")

if not looks_like_excel(r.content):
    raise RuntimeError(
        "Downloaded content does not look like Excel. "
        "Please confirm the repo/file is public and you are using raw.githubusercontent.com URL."
    )

with open(LOCAL_AAII, "wb") as f:
    f.write(r.content)

print(f"Saved to {LOCAL_AAII}")


# ===== Read Excel =====
xls = pd.ExcelFile(LOCAL_AAII)
print("\nAAII sheets:", xls.sheet_names)

sheet_name = "SENTIMENT" if "SENTIMENT" in xls.sheet_names else xls.sheet_names[0]

aaii_raw = pd.read_excel(LOCAL_AAII, sheet_name=sheet_name, header=None)

print("\nAAII raw preview:")
print(aaii_raw.head(15))


# ===== Auto-detect header row =====
header_idx = None

for i, row in aaii_raw.head(30).iterrows():
    row_text = " ".join(row.astype(str).str.lower().tolist())

    if "bullish" in row_text and "bearish" in row_text:
        header_idx = i
        print("\nHeader row found:", header_idx)
        print(row.values)
        break

if header_idx is None:
    raise ValueError("Could not find AAII header row. Please inspect aaii_raw.head(30).")


aaii = pd.read_excel(
    LOCAL_AAII,
    sheet_name=sheet_name,
    skiprows=header_idx
)

aaii.columns = [str(c).strip() for c in aaii.columns]

print("\nAAII columns:")
print(aaii.columns.tolist())
print(aaii.head())


# ===== Column mapping =====
col_map = {}

for c in aaii.columns:
    cl = str(c).lower().strip()

    if "date" in cl:
        col_map[c] = "date"
    elif "bullish" in cl and "average" not in cl and "high" not in cl and "low" not in cl:
        col_map[c] = "bullish"
    elif "neutral" in cl and "average" not in cl and "high" not in cl and "low" not in cl:
        col_map[c] = "neutral"
    elif "bearish" in cl and "average" not in cl and "high" not in cl and "low" not in cl:
        col_map[c] = "bearish"

print("\nColumn mapping:")
print(col_map)

aaii = aaii.rename(columns=col_map)

required_cols = ["date", "bullish", "neutral", "bearish"]
missing = [c for c in required_cols if c not in aaii.columns]

if missing:
    raise ValueError(
        f"Missing AAII columns: {missing}. "
        f"Current columns = {aaii.columns.tolist()}"
    )

aaii = aaii[required_cols].copy()
aaii = aaii.dropna(how="all")

aaii["date"] = pd.to_datetime(aaii["date"], errors="coerce")
aaii = aaii.dropna(subset=["date"])
aaii = aaii.set_index("date").sort_index()


def clean_pct(x):
    if isinstance(x, str):
        x = x.replace("%", "").strip()
    return pd.to_numeric(x, errors="coerce")


for col in ["bullish", "neutral", "bearish"]:
    aaii[col] = aaii[col].apply(clean_pct)

aaii = aaii.dropna(subset=["bullish", "neutral", "bearish"])

# 如果是 0-1 小数格式，转成 0-100 百分比
if aaii["bullish"].max() < 1.5:
    aaii[["bullish", "neutral", "bearish"]] *= 100

aaii["bb_spread"] = aaii["bullish"] - aaii["bearish"]

# 月度最后一期
aaii_features = aaii[["bb_spread"]].resample("ME").last().dropna()

print("\n=== AAII features ===")
print(aaii_features.tail(12))
print(aaii_features.describe().round(4))

aaii_features.to_csv("/content/aaii_features.csv")
print("\nSaved to /content/aaii_features.csv")

HTTP status: 200
Content-Type: application/octet-stream
File size KB: 1114.0
Saved to /content/sentiment.xls

AAII sheets: ['SENTIMENT', 'CHART']

AAII raw preview:
                     0        1        2   \
0                   NaN      NaN      NaN   
1                   NaN      NaN      NaN   
2              Reported      NaN      NaN   
3                  Date  Bullish  Neutral   
4                   NaN      NaN      NaN   
5   1987-06-26 00:00:00      NaN      NaN   
6   1987-07-17 00:00:00      NaN      NaN   
7   1987-07-24 00:00:00     0.36      0.5   
8   1987-07-31 00:00:00     0.26     0.48   
9   1987-08-07 00:00:00     0.56     0.15   
10  1987-08-14 00:00:00     0.45     0.35   
11  1987-08-21 00:00:00     0.66     0.28   
12  1987-08-28 00:00:00     0.52     0.18   
13  1987-09-04 00:00:00     0.42     0.17   
14  1987-09-11 00:00:00      0.5     0.23   

                                                   3      4        5   \
0   American Association of Individual In

In [21]:
# ============================================================
# Step J: Merge v3.2 features
# ============================================================

features_v32 = (
    price_features[
        [
            "dist_200dma",
            "tsmom_12m",
            "ret_1m",
            "ret_3m",
            "ret_6m",
            "dd_from_6m_high",
            "dd_from_12m_high",
            "vol_3m",
            "vol_ratio_3m_12m",
        ]
    ]
    .join(cape_features[["cape_z"]], how="left")
    .join(macro_features[["yc_10y3m", "cfnai_ma3"]], how="left")
    .join(aaii_features[["bb_spread"]], how="left")
)

features_v32 = features_v32.sort_index()

# Reorder columns
features_v32 = features_v32[
    [
        "cape_z",
        "dist_200dma",
        "tsmom_12m",
        "yc_10y3m",
        "cfnai_ma3",
        "bb_spread",
        "ret_1m",
        "ret_3m",
        "ret_6m",
        "dd_from_6m_high",
        "dd_from_12m_high",
        "vol_3m",
        "vol_ratio_3m_12m",
    ]
]

print("Features tail:")
print(features_v32.tail(12))

print("\nMissing values:")
print(features_v32.isna().sum())

print("\nFeature date range:")
print(features_v32.index.min(), "to", features_v32.index.max())

features_v32.to_csv("/content/features_v32.csv")
print("Saved to /content/features_v32.csv")

Features tail:
              cape_z  dist_200dma  tsmom_12m  yc_10y3m  cfnai_ma3  bb_spread  \
Date                                                                           
2025-06-30  1.723763     0.099376   0.152221     -0.17      -0.24    -5.2084   
2025-07-31  1.955230     0.106736   0.199133     -0.04      -0.12     7.3333   
2025-08-31  1.998539     0.099035   0.196212      0.00      -0.14    -4.8443   
2025-09-30  2.104706     0.140637   0.230266      0.14      -0.22     2.5180   
2025-10-31  2.186703     0.171801   0.300029      0.22      -0.39     7.1672   
2025-11-30  2.144328     0.135357   0.215215      0.14      -0.36   -10.6761   
2025-12-31  2.187015     0.101636   0.201677      0.51      -0.26     2.6431   
2026-01-31  2.162169     0.084099   0.189698      0.59      -0.07    13.6000   
2026-02-28  2.020953     0.035420   0.195152      0.30       0.03    -6.5574   
2026-03-31  1.654229    -0.028048   0.231437      0.60      -0.03   -17.6954   
2026-04-30  1.841170     

In [22]:
# ============================================================
# Step K: Generate M1 and M2 labels
# ============================================================

def future_stats_from_current(prices, current_idx, horizon_days=126):
    """
    Returns:
    1. future_6m_min_ret_from_current:
       Future 6M minimum return from current price.
    2. future_6m_peak_to_trough_mdd:
       Future 6M internal peak-to-trough max drawdown.
    """
    if current_idx + horizon_days >= len(prices):
        return np.nan, np.nan

    future_prices = prices.iloc[current_idx : current_idx + horizon_days + 1]
    current_price = future_prices.iloc[0]

    min_ret_from_current = future_prices.min() / current_price - 1

    running_max = future_prices.cummax()
    peak_to_trough_mdd = (future_prices / running_max - 1).min()

    return min_ret_from_current, peak_to_trough_mdd


month_ends = ndx_d.resample("ME").last().index

rows = []

for d in month_ends:
    if not (ndx_d.index <= d).any():
        rows.append((d, np.nan, np.nan))
        continue

    d_actual = ndx_d.index[ndx_d.index <= d][-1]
    idx = ndx_d.index.get_loc(d_actual)

    min_ret_current, p2t_mdd = future_stats_from_current(
        ndx_d["ndx_close"],
        idx,
        horizon_days=126
    )

    rows.append((d, min_ret_current, p2t_mdd))

labels_v32 = pd.DataFrame(
    rows,
    columns=[
        "date",
        "future_6m_min_ret_from_current",
        "future_6m_peak_to_trough_mdd",
    ]
).set_index("date")

labels_v32["label_m1_p2t_15"] = np.where(
    labels_v32["future_6m_peak_to_trough_mdd"].isna(),
    np.nan,
    (labels_v32["future_6m_peak_to_trough_mdd"] <= -0.15).astype(int)
)

labels_v32["label_m2_current_15"] = np.where(
    labels_v32["future_6m_min_ret_from_current"].isna(),
    np.nan,
    (labels_v32["future_6m_min_ret_from_current"] <= -0.15).astype(int)
)

print("Labels tail:")
print(labels_v32.tail(12))

print("\nMissing labels:")
print(labels_v32.isna().sum())

print("\nM1 base rate:")
print(labels_v32["label_m1_p2t_15"].dropna().mean())

print("\nM2 base rate:")
print(labels_v32["label_m2_current_15"].dropna().mean())

labels_v32.to_csv("/content/labels_v32.csv")
print("Saved to /content/labels_v32.csv")

Labels tail:
            future_6m_min_ret_from_current  future_6m_peak_to_trough_mdd  \
date                                                                       
2025-06-30                       -0.008857                     -0.079077   
2025-07-31                       -0.019589                     -0.079077   
2025-08-31                       -0.007871                     -0.079077   
2025-09-30                       -0.069960                     -0.121228   
2025-10-31                       -0.112334                     -0.117951   
2025-11-30                             NaN                           NaN   
2025-12-31                             NaN                           NaN   
2026-01-31                             NaN                           NaN   
2026-02-28                             NaN                           NaN   
2026-03-31                             NaN                           NaN   
2026-04-30                             NaN                           NaN   

In [23]:
# ============================================================
# Step L: Build M1 / M2 modeling tables
# ============================================================

base_model_features = [
    "cape_z",
    "dist_200dma",
    "tsmom_12m",
    "yc_10y3m",
    "cfnai_ma3",
    "bb_spread",
]

overlay_features = [
    "ret_1m",
    "ret_3m",
    "ret_6m",
    "dd_from_6m_high",
    "dd_from_12m_high",
    "vol_3m",
    "vol_ratio_3m_12m",
]

df_m1 = features_v32[base_model_features + overlay_features].join(
    labels_v32[["label_m1_p2t_15"]],
    how="inner"
).dropna(subset=base_model_features + ["label_m1_p2t_15"])

df_m1["label_15"] = df_m1["label_m1_p2t_15"].astype(int)

df_m2 = features_v32[base_model_features + overlay_features].join(
    labels_v32[["label_m2_current_15"]],
    how="inner"
).dropna(subset=base_model_features + ["label_m2_current_15"])

df_m2["label_15"] = df_m2["label_m2_current_15"].astype(int)

print("=== M1 table ===")
print("Start:", df_m1.index.min())
print("End:", df_m1.index.max())
print("Rows:", len(df_m1))
print("Label distribution:", df_m1["label_15"].value_counts().to_dict())
print("Base rate:", df_m1["label_15"].mean())

print("\n=== M2 table ===")
print("Start:", df_m2.index.min())
print("End:", df_m2.index.max())
print("Rows:", len(df_m2))
print("Label distribution:", df_m2["label_15"].value_counts().to_dict())
print("Base rate:", df_m2["label_15"].mean())

=== M1 table ===
Start: 1987-07-31 00:00:00
End: 2025-10-31 00:00:00
Rows: 460
Label distribution: {0: 321, 1: 139}
Base rate: 0.30217391304347824

=== M2 table ===
Start: 1987-07-31 00:00:00
End: 2025-10-31 00:00:00
Rows: 460
Label distribution: {0: 378, 1: 82}
Base rate: 0.1782608695652174


In [24]:
# ============================================================
# Step M: Walk-forward raw logistic for M1 / M2
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score

def walk_forward_raw_only(
    df,
    feature_cols,
    embargo_months=6,
    train_min_samples=60,
    train_min_positives=5,
    oos_start="2005-01-01",
    winsorize_quantiles=(0.01, 0.99),
    calibration_window_months=60,
):
    results = []
    all_dates = sorted(df.index.tolist())
    oos_start_ts = pd.Timestamp(oos_start)

    for cutoff in all_dates:
        if cutoff < oos_start_ts:
            continue

        cutoff_train_end = cutoff - pd.DateOffset(months=embargo_months)

        train_full = df[df.index < cutoff_train_end].copy()
        test_part = df[df.index == cutoff].copy()

        if len(train_full) < train_min_samples + calibration_window_months:
            continue

        if train_full["label_15"].sum() < train_min_positives:
            continue

        if len(test_part) == 0:
            continue

        bounds = {
            col: (
                train_full[col].quantile(winsorize_quantiles[0]),
                train_full[col].quantile(winsorize_quantiles[1]),
            )
            for col in feature_cols
        }

        train_w = train_full.copy()
        test_w = test_part.copy()

        for col in feature_cols:
            lo, hi = bounds[col]
            train_w[col] = train_w[col].clip(lo, hi)
            test_w[col] = test_w[col].clip(lo, hi)

        train_for_fit = train_w.iloc[:-calibration_window_months].copy()

        if train_for_fit["label_15"].sum() < train_min_positives:
            continue

        X_tr = train_for_fit[feature_cols]
        y_tr = train_for_fit["label_15"]

        scaler = StandardScaler().fit(X_tr)

        model = LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        ).fit(
            scaler.transform(X_tr),
            y_tr
        )

        p_raw = model.predict_proba(
            scaler.transform(test_w[feature_cols])
        )[:, 1][0]

        p_prior = train_full["label_15"].mean()

        results.append({
            "date": cutoff,
            "p_raw": float(p_raw),
            "p_prior": float(p_prior),
            "label": int(test_part["label_15"].iloc[0]),
        })

    return pd.DataFrame(results).set_index("date")


wf_m1 = walk_forward_raw_only(df_m1, base_model_features, oos_start="2005-01-01")
wf_m2 = walk_forward_raw_only(df_m2, base_model_features, oos_start="2005-01-01")

print("M1 walk-forward:")
print(wf_m1.head())
print(wf_m1.tail())

print("\nM2 walk-forward:")
print(wf_m2.head())
print(wf_m2.tail())


def safe_metrics(y, p):
    y = pd.Series(y).astype(int)
    p = pd.Series(p).astype(float)

    out = {
        "n": len(y),
        "base_rate": y.mean(),
        "brier": brier_score_loss(y, p),
        "baseline_brier_const": brier_score_loss(y, [y.mean()] * len(y)),
        "auc": np.nan,
        "pr_auc": np.nan,
        "p_mean": p.mean(),
        "p_std": p.std(),
        "p_min": p.min(),
        "p_max": p.max(),
    }

    if y.nunique() > 1:
        out["auc"] = roc_auc_score(y, p)
        out["pr_auc"] = average_precision_score(y, p)

    return out


metrics_m1 = safe_metrics(wf_m1["label"], wf_m1["p_raw"])
metrics_m2 = safe_metrics(wf_m2["label"], wf_m2["p_raw"])

print("\nM1 metrics:")
print(pd.Series(metrics_m1).round(4))

print("\nM2 metrics:")
print(pd.Series(metrics_m2).round(4))

wf_m1.to_csv("/content/wf_m1_v32.csv")
wf_m2.to_csv("/content/wf_m2_v32.csv")

M1 walk-forward:
               p_raw   p_prior  label
date                                 
2005-01-31  0.108470  0.362745      0
2005-02-28  0.110596  0.360976      0
2005-03-31  0.128523  0.359223      0
2005-04-30  0.118134  0.357488      0
2005-05-31  0.112298  0.355769      0
               p_raw   p_prior  label
date                                 
2025-06-30  0.349447  0.302895      0
2025-07-31  0.372645  0.304444      0
2025-08-31  0.377344  0.305987      0
2025-09-30  0.377234  0.307522      0
2025-10-31  0.416695  0.306843      0

M2 walk-forward:
               p_raw   p_prior  label
date                                 
2005-01-31  0.127522  0.215686      0
2005-02-28  0.132014  0.214634      0
2005-03-31  0.105661  0.213592      0
2005-04-30  0.111756  0.212560      0
2005-05-31  0.137182  0.211538      0
               p_raw   p_prior  label
date                                 
2025-06-30  0.270214  0.175947      0
2025-07-31  0.284947  0.177778      0
2025-08-31  0.2

In [25]:
# ============================================================
# Step N: Generate v3.2 online states
# ============================================================

def generate_v32_states(
    wf_m1,
    wf_m2,
    features,
    m1_q=0.80,
    m2_q=0.30,
    vol_confirm_th=1.00,
    min_history=60,
):
    state_df = pd.DataFrame(index=wf_m2.index)
    state_df.index = pd.to_datetime(state_df.index)

    state_df["m1_p_raw"] = wf_m1["p_raw"]
    state_df["m2_p_raw"] = wf_m2["p_raw"]
    state_df["label_current_15"] = wf_m2["label"]

    state_df = state_df.join(
        features[
            [
                "ret_1m",
                "ret_3m",
                "ret_6m",
                "dd_from_6m_high",
                "dd_from_12m_high",
                "vol_3m",
                "vol_ratio_3m_12m",
                "cape_z",
                "dist_200dma",
                "tsmom_12m",
                "yc_10y3m",
                "cfnai_ma3",
                "bb_spread",
            ]
        ],
        how="left"
    )

    state_df = state_df.dropna().copy()

    rows = []

    for i, date in enumerate(state_df.index):
        hist = state_df.iloc[:i]

        if len(hist) < min_history:
            rows.append((date, "WARMUP", np.nan, np.nan))
            continue

        m1_th = hist["m1_p_raw"].quantile(m1_q)
        m2_th = hist["m2_p_raw"].quantile(m2_q)

        row = state_df.loc[date]

        macro_shock = (
            (row["cfnai_ma3"] <= -1.0) |
            (row["vol_3m"] >= 0.35)
        )

        state = "NEUTRAL"

        if row["m2_p_raw"] <= m2_th:
            state = "GREEN_SAFE"

        red_warning = (
            (row["m1_p_raw"] >= m1_th) and
            (row["m2_p_raw"] >= m2_th) and
            (not macro_shock)
        )

        if red_warning:
            state = "RED_WARNING"

        if red_warning and row["vol_ratio_3m_12m"] >= vol_confirm_th:
            state = "RED_CONFIRM"

        rows.append((date, state, m1_th, m2_th))

    states = pd.DataFrame(
        rows,
        columns=["date", "state_v32", "m1_th_online", "m2_th_online"]
    ).set_index("date")

    return state_df.join(states, how="left")


final_state_v32 = generate_v32_states(
    wf_m1,
    wf_m2,
    features_v32,
    m1_q=0.80,
    m2_q=0.30,
    vol_confirm_th=1.00,
    min_history=60,
)

print("Final state tail:")
print(final_state_v32.tail(12))

eval_state = final_state_v32[final_state_v32["state_v32"] != "WARMUP"].copy()
base_rate = eval_state["label_current_15"].mean()

state_table = eval_state.groupby("state_v32").agg(
    n=("label_current_15", "size"),
    hit_rate=("label_current_15", "mean"),
    positives=("label_current_15", "sum"),
    avg_m1_p=("m1_p_raw", "mean"),
    avg_m2_p=("m2_p_raw", "mean"),
    avg_vol_ratio=("vol_ratio_3m_12m", "mean"),
    avg_cfnai=("cfnai_ma3", "mean"),
    avg_cape=("cape_z", "mean"),
).reset_index()

state_table["lift_vs_base"] = state_table["hit_rate"] / base_rate

state_order = {
    "GREEN_SAFE": 0,
    "NEUTRAL": 1,
    "RED_WARNING": 2,
    "RED_CONFIRM": 3,
}

state_table["order"] = state_table["state_v32"].map(state_order)
state_table = state_table.sort_values("order").drop(columns="order")

print("\nState table:")
print("Base rate:", round(base_rate, 4))
print(state_table.round(4))

final_state_v32.to_csv("/content/final_state_v32.csv")
print("Saved to /content/final_state_v32.csv")

Final state tail:
            m1_p_raw  m2_p_raw  label_current_15    ret_1m    ret_3m  \
date                                                                   
2024-11-30  0.495104  0.354390                 1  0.052284  0.069259   
2024-12-31  0.432362  0.304160                 1  0.003908  0.047430   
2025-01-31  0.405821  0.280593                 1  0.022172  0.079819   
2025-02-28  0.319721  0.244746                 1 -0.027639 -0.002196   
2025-03-31  0.395162  0.287885                 0 -0.076898 -0.082510   
2025-04-30  0.330811  0.210258                 0  0.015176 -0.088790   
2025-05-31  0.377789  0.256480                 0  0.090438  0.021862   
2025-06-30  0.349447  0.270214                 0  0.062697  0.176392   
2025-07-31  0.372645  0.284947                 0  0.023771  0.186352   
2025-08-31  0.377344  0.286900                 0  0.008498  0.097204   
2025-09-30  0.377234  0.292320                 0  0.054006  0.088231   
2025-10-31  0.416695  0.305757                

In [26]:
# ============================================================
# Step O: Live scoring for latest available month
# ============================================================

latest_feature_date = features_v32.dropna(subset=base_model_features).index.max()
latest_x = features_v32.loc[[latest_feature_date]]

print("Latest feature date:", latest_feature_date)
print(latest_x.T)


def fit_full_model_and_score_latest(df_train, feature_cols, latest_features):
    train = df_train.dropna(subset=feature_cols + ["label_15"]).copy()

    bounds = {
        col: (
            train[col].quantile(0.01),
            train[col].quantile(0.99)
        )
        for col in feature_cols
    }

    train_w = train.copy()
    latest_w = latest_features.copy()

    for col in feature_cols:
        lo, hi = bounds[col]
        train_w[col] = train_w[col].clip(lo, hi)
        latest_w[col] = latest_w[col].clip(lo, hi)

    X_tr = train_w[feature_cols]
    y_tr = train_w["label_15"]

    scaler = StandardScaler().fit(X_tr)

    model = LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42
    ).fit(
        scaler.transform(X_tr),
        y_tr
    )

    p = model.predict_proba(
        scaler.transform(latest_w[feature_cols])
    )[:, 1][0]

    return float(p)


live_m1_p = fit_full_model_and_score_latest(df_m1, base_model_features, latest_x)
live_m2_p = fit_full_model_and_score_latest(df_m2, base_model_features, latest_x)

hist_state = final_state_v32[final_state_v32["state_v32"] != "WARMUP"].copy()

m1_live_th = hist_state["m1_p_raw"].quantile(0.80)
m2_live_th = hist_state["m2_p_raw"].quantile(0.30)

latest_row = features_v32.loc[latest_feature_date]

macro_shock = (
    (latest_row["cfnai_ma3"] <= -1.0) |
    (latest_row["vol_3m"] >= 0.35)
)

live_state = "NEUTRAL"

if live_m2_p <= m2_live_th:
    live_state = "GREEN_SAFE"

red_warning = (
    (live_m1_p >= m1_live_th) and
    (live_m2_p >= m2_live_th) and
    (not macro_shock)
)

if red_warning:
    live_state = "RED_WARNING"

if red_warning and latest_row["vol_ratio_3m_12m"] >= 1.00:
    live_state = "RED_CONFIRM"

live_summary = pd.DataFrame([{
    "latest_feature_date": latest_feature_date,
    "live_state": live_state,

    "live_m1_p": live_m1_p,
    "live_m1_th": m1_live_th,

    "live_m2_p": live_m2_p,
    "live_m2_th": m2_live_th,

    "vol_ratio_3m_12m": latest_row["vol_ratio_3m_12m"],
    "vol_3m": latest_row["vol_3m"],
    "cfnai_ma3": latest_row["cfnai_ma3"],
    "cape_z": latest_row["cape_z"],
    "dist_200dma": latest_row["dist_200dma"],
    "tsmom_12m": latest_row["tsmom_12m"],
    "bb_spread": latest_row["bb_spread"],
}])

print("=== Live summary ===")
print(live_summary.T)

live_summary.to_csv("/content/live_summary_v32.csv", index=False)
print("Saved to /content/live_summary_v32.csv")

Latest feature date: 2026-03-31 00:00:00
Date              2026-03-31
cape_z              1.654229
dist_200dma        -0.028048
tsmom_12m           0.231437
yc_10y3m            0.600000
cfnai_ma3          -0.030000
bb_spread         -17.695400
ret_1m             -0.048872
ret_3m             -0.059789
ret_6m             -0.038079
dd_from_6m_high    -0.081906
dd_from_12m_high   -0.081906
vol_3m              0.182558
vol_ratio_3m_12m    0.802613
=== Live summary ===
                                       0
latest_feature_date  2026-03-31 00:00:00
live_state                   RED_WARNING
live_m1_p                        0.51753
live_m1_th                      0.360496
live_m2_p                       0.314243
live_m2_th                      0.092236
vol_ratio_3m_12m                0.802613
vol_3m                          0.182558
cfnai_ma3                          -0.03
cape_z                          1.654229
dist_200dma                    -0.028048
tsmom_12m                       0.231437

In [27]:
# ============================================================
# Step P: Backtest v3.2 default strategy with T-bill cash
# ============================================================

bt = final_state_v32[final_state_v32["state_v32"] != "WARMUP"].copy()
bt.index = pd.to_datetime(bt.index)

# NDX monthly return
ndx_m_close = ndx_d[["ndx_close"]].resample("ME").last()
ndx_m_close["ndx_ret_1m"] = ndx_m_close["ndx_close"].pct_change()

bt["next_month_ret"] = ndx_m_close["ndx_ret_1m"].shift(-1).reindex(bt.index)

# T-bill cash return
try:
    tb3ms
except NameError:
    tb3ms = fred.get_series("TB3MS", observation_start="2000-01-01")
    tb3ms = tb3ms.resample("ME").last().to_frame("tbill_3m_rate_pct")
    tb3ms["cash_ret_1m"] = tb3ms["tbill_3m_rate_pct"] / 100 / 12

bt["cash_next_month_ret"] = tb3ms["cash_ret_1m"].shift(-1).reindex(bt.index)
bt["cash_next_month_ret"] = bt["cash_next_month_ret"].fillna(0)

bt = bt.dropna(subset=["next_month_ret"]).copy()

# Exposure rules
bt["exposure_buy_hold"] = 1.0

bt["exposure_v32_confirm50"] = 1.0
bt.loc[bt["state_v32"] == "RED_CONFIRM", "exposure_v32_confirm50"] = 0.50

# Returns
for policy in ["buy_hold", "v32_confirm50"]:
    exp_col = f"exposure_{policy}"
    turnover = bt[exp_col].diff().abs().fillna(0)

    bt[f"ret_{policy}"] = (
        bt[exp_col] * bt["next_month_ret"]
        + (1 - bt[exp_col]) * bt["cash_next_month_ret"]
        - turnover * 0.0005
    )


def max_drawdown(equity):
    running_max = equity.cummax()
    dd = equity / running_max - 1
    return dd.min()


def perf_stats(ret, name):
    ret = ret.dropna()
    equity = (1 + ret).cumprod()

    years = len(ret) / 12
    cagr = equity.iloc[-1] ** (1 / years) - 1
    vol = ret.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else np.nan
    mdd = max_drawdown(equity)
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan

    worst_1m = ret.min()
    worst_6m = ((1 + ret).rolling(6).apply(np.prod, raw=True) - 1).min()

    return {
        "policy": name,
        "months": len(ret),
        "CAGR": cagr,
        "Vol": vol,
        "Sharpe_like": sharpe,
        "MaxDD": mdd,
        "Calmar": calmar,
        "Worst_1M": worst_1m,
        "Worst_6M": worst_6m,
        "Final_Wealth": equity.iloc[-1],
    }


perf_rows = [
    perf_stats(bt["ret_buy_hold"], "buy_hold"),
    perf_stats(bt["ret_v32_confirm50"], "v32_confirm50"),
]

perf_v32 = pd.DataFrame(perf_rows).set_index("policy")

display_perf = perf_v32.copy()
for c in ["CAGR", "Vol", "MaxDD", "Worst_1M", "Worst_6M"]:
    display_perf[c] *= 100

print("=== v3.2 backtest performance ===")
print(display_perf.round(2))

# State return profile
state_ret = bt.groupby("state_v32").agg(
    n=("next_month_ret", "size"),
    avg_next_ret=("next_month_ret", "mean"),
    median_next_ret=("next_month_ret", "median"),
    positive_rate=("next_month_ret", lambda x: (x > 0).mean()),
    worst_next_ret=("next_month_ret", "min"),
    best_next_ret=("next_month_ret", "max"),
)

print("\n=== Next-month return by state ===")
print(state_ret.round(4))

bt.to_csv("/content/v32_backtest_clean.csv")
perf_v32.to_csv("/content/v32_backtest_performance_clean.csv")

print("Saved:")
print("/content/v32_backtest_clean.csv")
print("/content/v32_backtest_performance_clean.csv")

=== v3.2 backtest performance ===
               months   CAGR    Vol  Sharpe_like  MaxDD  Calmar  Worst_1M  \
policy                                                                      
buy_hold          190  18.46  17.23         1.07 -32.97    0.56    -13.37   
v32_confirm50     190  18.58  15.90         1.17 -23.12    0.80     -9.06   

               Worst_6M  Final_Wealth  
policy                                 
buy_hold         -29.51         14.61  
v32_confirm50    -23.12         14.86  

=== Next-month return by state ===
              n  avg_next_ret  median_next_ret  positive_rate  worst_next_ret  \
state_v32                                                                       
GREEN_SAFE   42        0.0133           0.0230         0.6190         -0.0741   
NEUTRAL      79        0.0141           0.0112         0.6076         -0.0906   
RED_CONFIRM  31        0.0023           0.0037         0.5484         -0.1337   
RED_WARNING  38        0.0312           0.0396         0

In [28]:
# ============================================================
# Step P: Backtest v3.2 default strategy with T-bill cash
# ============================================================

bt = final_state_v32[final_state_v32["state_v32"] != "WARMUP"].copy()
bt.index = pd.to_datetime(bt.index)

# NDX monthly return
ndx_m_close = ndx_d[["ndx_close"]].resample("ME").last()
ndx_m_close["ndx_ret_1m"] = ndx_m_close["ndx_close"].pct_change()

bt["next_month_ret"] = ndx_m_close["ndx_ret_1m"].shift(-1).reindex(bt.index)

# T-bill cash return
try:
    tb3ms
except NameError:
    tb3ms = fred.get_series("TB3MS", observation_start="2000-01-01")
    tb3ms = tb3ms.resample("ME").last().to_frame("tbill_3m_rate_pct")
    tb3ms["cash_ret_1m"] = tb3ms["tbill_3m_rate_pct"] / 100 / 12

bt["cash_next_month_ret"] = tb3ms["cash_ret_1m"].shift(-1).reindex(bt.index)
bt["cash_next_month_ret"] = bt["cash_next_month_ret"].fillna(0)

bt = bt.dropna(subset=["next_month_ret"]).copy()

# Exposure rules
bt["exposure_buy_hold"] = 1.0

bt["exposure_v32_confirm50"] = 1.0
bt.loc[bt["state_v32"] == "RED_CONFIRM", "exposure_v32_confirm50"] = 0.50

# Returns
for policy in ["buy_hold", "v32_confirm50"]:
    exp_col = f"exposure_{policy}"
    turnover = bt[exp_col].diff().abs().fillna(0)

    bt[f"ret_{policy}"] = (
        bt[exp_col] * bt["next_month_ret"]
        + (1 - bt[exp_col]) * bt["cash_next_month_ret"]
        - turnover * 0.0005
    )


def max_drawdown(equity):
    running_max = equity.cummax()
    dd = equity / running_max - 1
    return dd.min()


def perf_stats(ret, name):
    ret = ret.dropna()
    equity = (1 + ret).cumprod()

    years = len(ret) / 12
    cagr = equity.iloc[-1] ** (1 / years) - 1
    vol = ret.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else np.nan
    mdd = max_drawdown(equity)
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan

    worst_1m = ret.min()
    worst_6m = ((1 + ret).rolling(6).apply(np.prod, raw=True) - 1).min()

    return {
        "policy": name,
        "months": len(ret),
        "CAGR": cagr,
        "Vol": vol,
        "Sharpe_like": sharpe,
        "MaxDD": mdd,
        "Calmar": calmar,
        "Worst_1M": worst_1m,
        "Worst_6M": worst_6m,
        "Final_Wealth": equity.iloc[-1],
    }


perf_rows = [
    perf_stats(bt["ret_buy_hold"], "buy_hold"),
    perf_stats(bt["ret_v32_confirm50"], "v32_confirm50"),
]

perf_v32 = pd.DataFrame(perf_rows).set_index("policy")

display_perf = perf_v32.copy()
for c in ["CAGR", "Vol", "MaxDD", "Worst_1M", "Worst_6M"]:
    display_perf[c] *= 100

print("=== v3.2 backtest performance ===")
print(display_perf.round(2))

# State return profile
state_ret = bt.groupby("state_v32").agg(
    n=("next_month_ret", "size"),
    avg_next_ret=("next_month_ret", "mean"),
    median_next_ret=("next_month_ret", "median"),
    positive_rate=("next_month_ret", lambda x: (x > 0).mean()),
    worst_next_ret=("next_month_ret", "min"),
    best_next_ret=("next_month_ret", "max"),
)

print("\n=== Next-month return by state ===")
print(state_ret.round(4))

bt.to_csv("/content/v32_backtest_clean.csv")
perf_v32.to_csv("/content/v32_backtest_performance_clean.csv")

print("Saved:")
print("/content/v32_backtest_clean.csv")
print("/content/v32_backtest_performance_clean.csv")

=== v3.2 backtest performance ===
               months   CAGR    Vol  Sharpe_like  MaxDD  Calmar  Worst_1M  \
policy                                                                      
buy_hold          190  18.46  17.23         1.07 -32.97    0.56    -13.37   
v32_confirm50     190  18.58  15.90         1.17 -23.12    0.80     -9.06   

               Worst_6M  Final_Wealth  
policy                                 
buy_hold         -29.51         14.61  
v32_confirm50    -23.12         14.86  

=== Next-month return by state ===
              n  avg_next_ret  median_next_ret  positive_rate  worst_next_ret  \
state_v32                                                                       
GREEN_SAFE   42        0.0133           0.0230         0.6190         -0.0741   
NEUTRAL      79        0.0141           0.0112         0.6076         -0.0906   
RED_CONFIRM  31        0.0023           0.0037         0.5484         -0.1337   
RED_WARNING  38        0.0312           0.0396         0